# Ingestão SGS -> Bronze (Fabric Lakehouse)

Adaptado de `ingestion/ingestir_sgs.py` (versão local DuckDB) para rodar
como Notebook Fabric (PySpark), escrevendo em `bronze.sgs_series_raw`
no Lakehouse anexado a este notebook.

**Antes de rodar**: anexe um Lakehouse com **schemas habilitados**
(opção 'Enable schemas (preview)' na criação do Lakehouse) a este
notebook, para manter a mesma convenção bronze/silver/gold do projeto.

In [ ]:
import time
from datetime import datetime, timedelta, timezone

import requests
from pyspark.sql import Row

SERIES_SGS = {
    "selic_diaria": 11,
    "selic_meta": 432,
    "ipca_mensal": 433,
    "saldo_credito_total": 20539,
    "credito_pib": 20622,
    "inadimplencia_total": 21082,
    "spread_medio_total": 20783,
    "concessoes_pf_total": 20633,
    "concessoes_pj_total": 20632,
    "endividamento_familias": 29037,
}

DATA_INICIAL_COMPLETA = "01/01/2015"
HOJE = datetime.now(timezone.utc).date()
DATA_INICIAL_JANELA_10_ANOS = (HOJE - timedelta(days=365 * 10 - 1)).strftime("%d/%m/%Y")
DATA_FINAL = HOJE.strftime("%d/%m/%Y")

MAX_TENTATIVAS = 4
TIMEOUT_SEGUNDOS = 30

In [ ]:
def chamar_api(codigo, data_inicial):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    parametros = {"formato": "json", "dataInicial": data_inicial, "dataFinal": DATA_FINAL}
    resposta = requests.get(url, params=parametros, timeout=TIMEOUT_SEGUNDOS)
    return resposta, url


def chamar_api_com_retry(codigo, data_inicial):
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            resposta, url = chamar_api(codigo, data_inicial)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as erro:
            espera = 2 ** (tentativa - 1)
            print(f"tentativa {tentativa} falhou (falha de rede: {erro}), esperando {espera}s...")
            time.sleep(espera)
            continue

        if resposta.status_code == 200:
            try:
                resposta.json()
                return resposta, url
            except requests.exceptions.JSONDecodeError:
                pass

        if resposta.status_code == 406:
            return resposta, url

        espera = 2 ** (tentativa - 1)
        motivo = "JSON inválido" if resposta.status_code == 200 else f"status {resposta.status_code}"
        print(f"tentativa {tentativa} falhou ({motivo}), esperando {espera}s...")
        time.sleep(espera)

    raise RuntimeError(f"Falhou após {MAX_TENTATIVAS} tentativas para código {codigo}")


def validar_schema_resposta(dados, nome_serie):
    if not isinstance(dados, list):
        raise TypeError(f"{nome_serie}: esperava uma lista, recebeu {type(dados)}")
    if len(dados) == 0:
        raise ValueError(f"{nome_serie}: resposta vazia, sem registros")
    chaves_esperadas = {"data", "valor"}
    chaves_recebidas = set(dados[0].keys())
    if chaves_recebidas != chaves_esperadas:
        raise ValueError(
            f"{nome_serie}: schema mudou! Esperado {chaves_esperadas}, recebido {chaves_recebidas}"
        )


def buscar_serie(nome, codigo):
    resposta, url = chamar_api_com_retry(codigo, DATA_INICIAL_COMPLETA)
    if resposta.status_code == 406:
        print(f"{nome}: periodicidade diária detectada, ajustando janela...")
        resposta, url = chamar_api_com_retry(codigo, DATA_INICIAL_JANELA_10_ANOS)
    resposta.raise_for_status()
    dados = resposta.json()
    validar_schema_resposta(dados, nome)
    print(f"{nome} (código {codigo}): {len(dados)} registros")
    return dados, url


def montar_linhas(nome, codigo, dados, url):
    timestamp_coleta = datetime.now(timezone.utc).isoformat()
    linhas = []
    for registro in dados:
        linhas.append(Row(
            nome_serie=nome,
            codigo_serie=codigo,
            data_referencia=registro["data"],
            valor=registro["valor"],
            url_fonte=url,
            timestamp_coleta=timestamp_coleta,
        ))
    return linhas

In [ ]:
todas_as_linhas = []
for nome, codigo in SERIES_SGS.items():
    dados, url = buscar_serie(nome, codigo)
    todas_as_linhas.extend(montar_linhas(nome, codigo, dados, url))

print(f"Total de linhas a inserir: {len(todas_as_linhas)}")

df = spark.createDataFrame(todas_as_linhas)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Bronze é append-only: nunca apagamos dados existentes (ver ADR-003).
df.write.format("delta").mode("append").saveAsTable("bronze.sgs_series_raw")

total_na_tabela = spark.sql("SELECT COUNT(*) AS total FROM bronze.sgs_series_raw").collect()[0]["total"]
print(f"Total de linhas agora na tabela: {total_na_tabela}")